# Mixed Preference BO：複数ペアの逐次最適化

連続変数とカテゴリ変数を含む選好学習です。1ラウンドに複数の独立した2候補比較を行います。比較の順序は `[winner, loser]` です。

## このNotebookで学ぶこと

- 連続変数とカテゴリ変数を同時に扱うMixed Preference BOの考え方
- `MixedPairwiseGP` の学習と、比較データ `[winner, loser]` の意味
- **EUBO（Expected Utility of Best Option）** で2候補の比較ペアを提案する仕組み
- `pairs_per_step=2` による**独立した2ペアの同時評価**と `q>2` joint最適化の違い
- 比較結果を履歴に追加して再学習する一連の流れ
- ラウンドごとの提案点と比較結果の可視化

合成効用はデモ用です。実際の運用では、勝者を返す `feedback` を人間・実験の結果に置き換えます。

In [ ]:
import torch

from robotorchan.optim.preference_loop import run_mixed_preference_batch_loop

torch.manual_seed(0)
dtype = torch.double

## 1. 初期比較データ

第1列は連続値、第2列はカテゴリ（0または1）です。

### Mixed入力と比較インデックス

入力は `x = [x_continuous, category]` の2次元です。`cat_dims=[1]` により第2列をカテゴリとして扱い、`fixed_features_list` でカテゴリ0・1を列挙します。カテゴリを0.5のような中間値として探索しません。

`comparisons[k] = [i, j]` は、`datapoints[i]` が `datapoints[j]` より好まれたことを表します。値そのものではなく、**行番号**を渡す点に注意してください。

In [ ]:
datapoints = torch.tensor(
    [[0.0, 0.0], [0.3, 1.0], [0.7, 0.0], [1.0, 1.0]], dtype=dtype
)
comparisons = torch.tensor([[1, 0], [2, 3], [1, 2]], dtype=torch.long)
bounds = torch.tensor([[0.0, 0.0], [1.0, 1.0]], dtype=dtype)
fixed_features_list = [{1: 0.0}, {1: 1.0}]

In [ ]:
print("initial candidates:")
for idx, (continuous, category) in enumerate(datapoints.tolist()):
    print(f"  item {idx}: x={continuous:.2f}, category={int(category)}")
print("winner/loser comparisons:", comparisons.tolist())
assert comparisons.dtype == torch.long
assert comparisons.min() >= 0
assert comparisons.max() < len(datapoints)
assert all(int(x) in (0, 1) for x in datapoints[:, 1].tolist())

## 2. 比較フィードバック

実際の運用では人間・実験から各ペアの勝者を受け取ります。ここでは再現可能な疑似utilityを使います。

### 最新のモデル学習API

`run_mixed_preference_batch_loop()` は内部で各ラウンドごとに `MixedPairwiseGP(points, preferences, cat_dims=cat_dims)` を構築し、公開 `model.fit(optimizer_kwargs=...)` を呼び出します。

`MixedPairwiseGP` は `PairwiseGP` の派生モデルで、比較データに対する**Pairwise Laplace MLL**を学習します。`fit_gpytorch_mll()` をNotebookから直接呼ぶ必要はありません。学習後は `AnalyticExpectedUtilityOfBestOption(pref_model=model)` を使ってEUBOを計算します。

In [ ]:
def feedback(pairs: torch.Tensor) -> list[int]:
    utility = torch.sin(2 * torch.pi * pairs[..., 0]) + 0.2 * pairs[..., 1]
    return utility.argmax(dim=1).tolist()

### `feedback` の入出力

`pairs` の形状は `(pairs_per_step, 2, d)` です。この例では `(2, 2, 2)` となり、各ペアの2候補から**どちらが好ましいか**をインデックス0または1で返します。

ここでは `sin(2πx_continuous) + 0.2*category` を疑似効用として使います。効用値自体をモデルへ渡すのではなく、**勝者のインデックスだけ**を学習履歴へ追加します。

## 3. 逐次最適化

各ラウンドで1回モデルを学習し、独立したq=2ペアを2組提案します。重複時にはランダム候補からEUBOの高い代替ペアを選びます。

### EUBOによる複数ペアの提案

1ラウンドの処理は次の順序です。

1. 現在の比較履歴で `MixedPairwiseGP` を1回学習する
2. EUBOを用いて `q=2` の比較ペアを最適化する
3. `pairs_per_step=2` の場合、同じ学習済みモデルから**独立した2ペア**を提案する
4. ペアが重複した場合は、ランダム候補プールからEUBOの高い代替ペアを選ぶ
5. `feedback` で各ペアの勝者を得る
6. 新しい候補4点と比較2件を履歴に追加し、次のラウンドで再学習する

**重要：** これは4候補をまとめて最適化するjoint `q=4` 獲得関数ではありません。`min_pair_distance` は同一ラウンド内のペア間の重複・近接を抑えるための設定です。既存観測点との重複回避は保証しません。

In [ ]:
updated_points, updated_comparisons, batches = run_mixed_preference_batch_loop(
    datapoints,
    comparisons,
    cat_dims=[1],
    bounds=bounds,
    fixed_features_list=fixed_features_list,
    feedback=feedback,
    pairs_per_step=2,
    steps=2,
    min_pair_distance=0.05,
    diversity_samples=64,
    fit_optimizer_kwargs={"options": {"maxiter": 10}},
    num_restarts=2,
    raw_samples=8,
    options={"maxiter": 10},
)
print("datapoints:", updated_points.shape)
print("comparisons:", updated_comparisons.shape)
for step, batch in enumerate(batches, start=1):
    print(f"round {step}:", batch)

### 履歴の増加を確認

各ラウンドで2ペア（4候補、2比較）を追加します。2ラウンド終了後は、初期4候補・3比較から、12候補・7比較になります。新しい比較は常に追加した候補の行番号を参照します。

In [ ]:
expected_points = len(datapoints) + 2 * 2 * 2
expected_comparisons = len(comparisons) + 2 * 2
assert updated_points.shape == (expected_points, 2)
assert updated_comparisons.shape == (expected_comparisons, 2)
assert len(batches) == 2
assert all(batch.shape == (2, 2, 2) for batch in batches)
assert updated_comparisons.max().item() < len(updated_points)
assert torch.all((updated_points[:, 1] == 0) | (updated_points[:, 1] == 1))
print("initial:", len(datapoints), "items /", len(comparisons), "comparisons")
print("updated:", len(updated_points), "items /", len(updated_comparisons), "comparisons")
print("new comparisons:", updated_comparisons[len(comparisons):].tolist())

### 提案された比較ペアを可視化

横軸は連続変数、縦軸はカテゴリです。同じ線で結んだ2点が1つの比較ペアを表します。カテゴリは数値順序のある連続量ではなく、識別子として表示しています。

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(batches), figsize=(11, 4), sharex=True, sharey=True)
for step, (ax, batch) in enumerate(zip(axes, batches), start=1):
    for pair_idx, pair in enumerate(batch):
        values = pair.detach().cpu()
        ax.plot(values[:, 0], values[:, 1], "o-", label=f"pair {pair_idx + 1}")
        ax.annotate("A", (values[0, 0].item(), values[0, 1].item()), xytext=(0, 7),
                    textcoords="offset points", ha="center")
        ax.annotate("B", (values[1, 0].item(), values[1, 1].item()), xytext=(0, 7),
                    textcoords="offset points", ha="center")
    ax.set(title=f"round {step}", xlabel="continuous x", xlim=(-0.05, 1.05),
           yticks=[0, 1], yticklabels=["category 0", "category 1"])
    ax.legend()
axes[0].set_ylabel("category")
plt.tight_layout()
plt.show()

## 4. 注意事項

- 各ペアはq=2のAnalytic EUBOで評価します。q>2のjoint EUBOやlistwise順位学習ではありません。
- 同一ラウンドの重複を回避しますが、既存観測点との重複回避は保証しません。
- カテゴリの組合せが多い場合、列挙コストが増加します。
- 実データで利用する場合は`feedback`を評価・実験結果の取得処理に置き換えます。

### 運用時の注意点

- `feedback` は各ペアの勝者を示すPython整数 `0` または `1` を、ペア数と同じ長さのリストで返します
- `fixed_features_list` は許容するカテゴリの組合せを列挙します。カテゴリ数が増えると最適化コストも増加します
- 同一ラウンドの各ペアは**同じPosterior**から選ばれます。ラウンド内で途中のフィードバックを反映するわけではありません
- 返却される `updated_points` と `updated_comparisons` を保存すれば、次回の実行で履歴を再利用できます
- 実験や官能評価では比較のばらつき、評価者差、コストも検討してください

## まとめ：学習から比較結果の更新まで

| 段階 | 使用するAPI・データ |
|---|---|
| 初期観測 | `datapoints`、`comparisons`（勝者・敗者の行番号） |
| Mixedモデル | `MixedPairwiseGP(cat_dims=[1])`（ループ内部） |
| 学習 | `model.fit(optimizer_kwargs=...)`（ループ内部） |
| 獲得関数 | `AnalyticExpectedUtilityOfBestOption`（EUBO） |
| Mixed候補最適化 | `optimize_mixed_preference_pair()`（ループ内部） |
| 複数ペア提案 | `run_mixed_preference_batch_loop(pairs_per_step=2)` |
| 観測 | `feedback(batch)` が各ペアの勝者を返す |
| 履歴更新 | `updated_points`、`updated_comparisons` |

**このNotebookは事後平均だけの候補選択ではありません。** EUBOによるMixed候補の提案、比較フィードバック、履歴追加、再学習までを既存の公開APIで実行します。